# Airport Analysis
This notebook extracts and analyzes all distinct airports from the historical flights data.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# Load historical flights data
data_path = Path("../data")
df_hist = pd.read_csv(data_path / "avinor_data/historical_flights.csv")

print(f"✅ Historical flights data loaded successfully")
print(f"📊 Dataset shape: {df_hist.shape}")
print(f"📋 Columns: {list(df_hist.columns)}")
print(f"\n🔍 First few rows:")
print(df_hist.head())

✅ Historical flights data loaded successfully
📊 Dataset shape: (410442, 11)
📋 Columns: ['flight_id', 'dep_airport', 'dep_airport_group', 'arr_airport', 'arr_airport_group', 'service_type', 'std', 'sta', 'cancelled', 'atd', 'ata']

🔍 First few rows:
  flight_id dep_airport dep_airport_group arr_airport arr_airport_group  \
0     WF149         HOV                 B         OSL               NaN   
1     WF722         OSL               NaN         MJF                 D   
2     WF188         FDE                 A         OSL               NaN   
3     WF176         HOV                 B         OSL               NaN   
4     WF148         HOV                 B         OSL               NaN   

  service_type                         std                         sta  \
0            J  2018-01-02T16:40:00.000000  2018-01-02T17:15:00.000000   
1            J  2018-01-28T13:04:00.000000  2018-01-28T14:50:00.000000   
2            J  2018-04-07T07:10:00.000000  2018-04-07T08:10:00.000000   
3   

In [ ]:
# Extract all distinct airports
print("🛫 Extracting distinct airports from historical flights data...")

# Get unique departure airports
dep_airports = set(df_hist["dep_airport"].dropna().unique())
print(f"📍 Unique departure airports: {len(dep_airports)}")

# Get unique arrival airports
arr_airports = set(df_hist["arr_airport"].dropna().unique())
print(f"📍 Unique arrival airports: {len(arr_airports)}")

# Combine all airports (departure + arrival)
all_airports = sorted(list(dep_airports.union(arr_airports)))
print(f"✈️  Total distinct airports: {len(all_airports)}")

print(f"\n📋 Complete list of all distinct airports:")
print("=" * 50)
for i, airport in enumerate(all_airports, 1):
    print(f"{i:2d}. {airport}")

print(f"\n📊 Summary:")
print(f"   • Total airports: {len(all_airports)}")
print(f"   • Departure-only airports: {len(dep_airports - arr_airports)}")
print(f"   • Arrival-only airports: {len(arr_airports - dep_airports)}")
print(
    f"   • Airports serving both dep/arr: {len(dep_airports.intersection(arr_airports))}"
)

# Store the list for further use
airports_list = all_airports
print(f"\n✅ Airports list stored in 'airports_list' variable")

🛫 Extracting distinct airports from historical flights data...
📍 Unique departure airports: 118
📍 Unique arrival airports: 122
✈️  Total distinct airports: 140

📋 Complete list of all distinct airports:
 1. 1HR
 2. 1SN
 3. 3SK
 4. AAL
 5. AAR
 6. ABZ
 7. ACE
 8. AES
 9. AGP
10. ALC
11. ALF
12. AMS
13. ANX
14. AOK
15. ARN
16. AYT
17. BCN
18. BDU
19. BEG
20. BFS
21. BGO
22. BGY
23. BHX
24. BJF
25. BJJ
26. BLL
27. BMA
28. BNN
29. BOO
30. BQH
31. BRN
32. BRU
33. BUD
34. BVG
35. CDT
36. CGN
37. CHQ
38. CPH
39. DEB
40. DUB
41. EDI
42. EVE
43. EXT
44. FAE
45. FAO
46. FDE
47. FRA
48. FRO
49. GDN
50. GOT
51. GVA
52. GYD
53. HAA
54. HAM
55. HAU
56. HEL
57. HFT
58. HOV
59. HVG
60. IST
61. JKG
62. KEF
63. KKN
64. KLU
65. KRS
66. KSU
67. KTW
68. LAX
69. LBG
70. LCA
71. LGG
72. LGW
73. LIL
74. LJU
75. LKL
76. LKN
77. LLA
78. LPA
79. LPI
80. LPL
81. LTN
82. MBX
83. MEH
84. MJF
85. MMX
86. MOL
87. MQN
88. MUC
89. NHT
90. OER
91. OLA
92. OSD
93. OSL
94. OST
95. OSY
96. OVD
97. PAD
98. PEK
99. PMI
100. 

## Airport Coordinates Lookup
Finding latitude and longitude coordinates for each airport using available APIs and databases.

In [4]:
# Install required packages for API requests
import requests
import json
import time
from typing import Dict, Tuple, Optional

print("🔧 Required packages imported for API requests")

🔧 Required packages imported for API requests


In [ ]:
# Functions to fetch airport coordinates from multiple sources


def get_airport_coordinates_from_aviation_api(
    iata_code: str,
) -> Optional[Tuple[float, float]]:
    """
    Fetch airport coordinates using aviation-edge API (free tier available)
    Returns (latitude, longitude) or None if not found
    """
    try:
        # Using a free aviation API - no key required for basic airport data
        url = (
            f"https://aviation-edge.com/v2/public/airportDatabase?iataCode={iata_code}"
        )
        response = requests.get(url, timeout=10)

        if response.status_code == 200:
            data = response.json()
            if data and len(data) > 0:
                airport_info = data[0]
                lat = float(airport_info.get("latitudeAirport", 0))
                lon = float(airport_info.get("longitudeAirport", 0))
                if lat != 0 and lon != 0:
                    return (lat, lon)
    except Exception as e:
        print(f"   ⚠️ Aviation API error for {iata_code}: {str(e)[:50]}")

    return None


def get_airport_coordinates_from_nominatim(
    iata_code: str,
) -> Optional[Tuple[float, float]]:
    """
    Fetch airport coordinates using Nominatim (OpenStreetMap) geocoding
    Searches for "airport_code airport"
    """
    try:
        # Use Nominatim API (free OpenStreetMap geocoding)
        search_query = f"{iata_code} airport"
        url = "https://nominatim.openstreetmap.org/search"
        params = {"q": search_query, "format": "json", "limit": 1, "addressdetails": 1}
        headers = {"User-Agent": "AirportAnalysis/1.0"}

        response = requests.get(url, params=params, headers=headers, timeout=10)

        if response.status_code == 200:
            data = response.json()
            if data and len(data) > 0:
                result = data[0]
                lat = float(result["lat"])
                lon = float(result["lon"])
                return (lat, lon)
    except Exception as e:
        print(f"   ⚠️ Nominatim error for {iata_code}: {str(e)[:50]}")

    return None


def get_airport_coordinates_multiple_sources(iata_code: str) -> Optional[Dict]:
    """
    Try multiple sources to get airport coordinates
    Returns dict with coordinates and source info
    """
    print(f"🔍 Looking up coordinates for {iata_code}...")

    # Try aviation API first
    coords = get_airport_coordinates_from_aviation_api(iata_code)
    if coords:
        return {
            "iata_code": iata_code,
            "latitude": coords[0],
            "longitude": coords[1],
            "source": "aviation-edge",
        }

    # Try Nominatim as backup
    time.sleep(1)  # Be respectful to free APIs
    coords = get_airport_coordinates_from_nominatim(iata_code)
    if coords:
        return {
            "iata_code": iata_code,
            "latitude": coords[0],
            "longitude": coords[1],
            "source": "nominatim",
        }

    print(f"   ❌ No coordinates found for {iata_code}")
    return {
        "iata_code": iata_code,
        "latitude": None,
        "longitude": None,
        "source": "not_found",
    }


print("✅ Airport coordinate lookup functions created")

✅ Airport coordinate lookup functions created


In [ ]:
# Test coordinate lookup with a few airports first
print("🧪 Testing coordinate lookup with first 5 airports...")

test_airports = airports_list[:5]
print(f"Testing with: {test_airports}")

for airport_code in test_airports:
    coord_info = get_airport_coordinates_multiple_sources(airport_code)
    if coord_info["latitude"] is not None:
        print(
            f"✅ {airport_code}: {coord_info['latitude']:.4f}, {coord_info['longitude']:.4f} ({coord_info['source']})"
        )
    else:
        print(f"❌ {airport_code}: No coordinates found")
    time.sleep(1)  # Be respectful to APIs

print("\n✅ Test complete! Functions are working.")

🧪 Testing coordinate lookup with first 5 airports...
Testing with: ['1HR', '1SN', '3SK', 'AAL', 'AAR']
🔍 Looking up coordinates for 1HR...
   ⚠️ Aviation API error for 1HR: 0
✅ 1HR: 60.3188, 24.9681 (nominatim)
🔍 Looking up coordinates for 1SN...
   ⚠️ Aviation API error for 1SN: 0
✅ 1SN: 60.3188, 24.9681 (nominatim)
🔍 Looking up coordinates for 3SK...
   ⚠️ Aviation API error for 3SK: 0
   ❌ No coordinates found for 3SK
❌ 3SK: No coordinates found
🔍 Looking up coordinates for AAL...
   ⚠️ Aviation API error for AAL: 0
✅ AAL: 57.0864, 9.8680 (nominatim)
🔍 Looking up coordinates for AAR...
   ⚠️ Aviation API error for AAR: 0
✅ AAR: 56.3047, 10.6123 (nominatim)

✅ Test complete! Functions are working.


In [ ]:
# Fetch coordinates for all airports
print("🌍 Starting coordinate lookup for all airports...")
print(f"📊 Total airports to process: {len(airports_list)}")
print("=" * 60)

airport_coordinates = []
successful_lookups = 0
failed_lookups = 0

# Process airports in batches to be respectful to APIs
for i, airport_code in enumerate(airports_list, 1):
    print(f"[{i:3d}/{len(airports_list)}] Processing {airport_code}...")

    # Get coordinates
    coord_info = get_airport_coordinates_multiple_sources(airport_code)
    airport_coordinates.append(coord_info)

    if coord_info["latitude"] is not None:
        successful_lookups += 1
        print(
            f"   ✅ Found: {coord_info['latitude']:.4f}, {coord_info['longitude']:.4f} ({coord_info['source']})"
        )
    else:
        failed_lookups += 1

    # Add delay every 10 requests to be respectful to free APIs
    if i % 10 == 0:
        print(
            f"   ⏳ Pausing briefly... ({successful_lookups} found, {failed_lookups} failed so far)"
        )
        time.sleep(2)

print("\n" + "=" * 60)
print(f"🎯 COORDINATE LOOKUP COMPLETE!")
print(f"   ✅ Successful lookups: {successful_lookups}")
print(f"   ❌ Failed lookups: {failed_lookups}")
print(f"   📊 Success rate: {(successful_lookups/len(airports_list)*100):.1f}%")

🌍 Starting coordinate lookup for all airports...
📊 Total airports to process: 140
[  1/140] Processing 1HR...
🔍 Looking up coordinates for 1HR...
   ⚠️ Aviation API error for 1HR: 0
   ✅ Found: 60.3188, 24.9681 (nominatim)
[  2/140] Processing 1SN...
🔍 Looking up coordinates for 1SN...
   ⚠️ Aviation API error for 1SN: 0
   ✅ Found: 60.3188, 24.9681 (nominatim)
[  3/140] Processing 3SK...
🔍 Looking up coordinates for 3SK...
   ⚠️ Aviation API error for 3SK: 0
   ❌ No coordinates found for 3SK
[  4/140] Processing AAL...
🔍 Looking up coordinates for AAL...
   ⚠️ Aviation API error for AAL: 0
   ✅ Found: 57.0864, 9.8680 (nominatim)
[  5/140] Processing AAR...
🔍 Looking up coordinates for AAR...
   ⚠️ Aviation API error for AAR: 0
   ✅ Found: 56.3047, 10.6123 (nominatim)
[  6/140] Processing ABZ...
🔍 Looking up coordinates for ABZ...
   ⚠️ Aviation API error for ABZ: 0
   ✅ Found: 57.2022, -2.2017 (nominatim)
[  7/140] Processing ACE...
🔍 Looking up coordinates for ACE...
   ⚠️ Aviation A

In [ ]:
# Create DataFrame with airport coordinates
df_airports = pd.DataFrame(airport_coordinates)

print("📋 AIRPORT COORDINATES SUMMARY:")
print("=" * 50)

# Display first few results
print("\n🔍 First 10 airports with coordinates:")
df_with_coords = df_airports[df_airports["latitude"].notna()].head(10)
for _, row in df_with_coords.iterrows():
    print(
        f"   {row['iata_code']}: {row['latitude']:.4f}, {row['longitude']:.4f} ({row['source']})"
    )

# Display airports without coordinates
print(f"\n❌ Airports without coordinates ({failed_lookups} total):")
df_no_coords = df_airports[df_airports["latitude"].isna()]
no_coords_list = df_no_coords["iata_code"].tolist()
print(f"   {', '.join(no_coords_list[:20])}")
if len(no_coords_list) > 20:
    print(f"   ... and {len(no_coords_list) - 20} more")

# Summary statistics
print(f"\n📊 COORDINATE STATISTICS:")
print(f"   • Total airports: {len(df_airports)}")
print(f"   • With coordinates: {len(df_airports[df_airports['latitude'].notna()])}")
print(f"   • Without coordinates: {len(df_airports[df_airports['latitude'].isna()])}")
print(
    f"   • Success rate: {(len(df_airports[df_airports['latitude'].notna()])/len(df_airports)*100):.1f}%"
)

# Source breakdown
source_counts = df_airports["source"].value_counts()
print(f"\n🔍 SOURCES USED:")
for source, count in source_counts.items():
    print(f"   • {source}: {count} airports")

print(f"\n✅ Airport coordinates DataFrame created as 'df_airports'")
print(f"📁 Ready for export or further analysis")

📋 AIRPORT COORDINATES SUMMARY:

🔍 First 10 airports with coordinates:
   1HR: 60.3188, 24.9681 (nominatim)
   1SN: 60.3188, 24.9681 (nominatim)
   AAL: 57.0864, 9.8680 (nominatim)
   AAR: 56.3047, 10.6123 (nominatim)
   ABZ: 57.2022, -2.2017 (nominatim)
   ACE: 28.9457, -13.6076 (nominatim)
   AES: 62.5602, 6.1134 (nominatim)
   AGP: 36.6785, -4.5000 (nominatim)
   ALC: 38.2844, -0.5658 (nominatim)
   ALF: 69.9772, 23.3595 (nominatim)

❌ Airports without coordinates (4 total):
   3SK, SXF, VDB, VRY

📊 COORDINATE STATISTICS:
   • Total airports: 140
   • With coordinates: 136
   • Without coordinates: 4
   • Success rate: 97.1%

🔍 SOURCES USED:
   • nominatim: 136 airports
   • not_found: 4 airports

✅ Airport coordinates DataFrame created as 'df_airports'
📁 Ready for export or further analysis


In [ ]:
# Save results and display final DataFrame
print("💾 Saving airport coordinates to CSV...")

# Save to CSV file
output_file = "../data/airport_coordinates.csv"
df_airports.to_csv(output_file, index=False)
print(f"✅ Saved to: {output_file}")

# Display the complete DataFrame
print(f"\n📋 COMPLETE AIRPORT COORDINATES TABLE:")
print("=" * 80)
print(df_airports.to_string(index=False))

# Quick validation - show coordinate ranges
coords_df = df_airports[df_airports["latitude"].notna()]
if len(coords_df) > 0:
    print(f"\n🌍 COORDINATE RANGES:")
    print(
        f"   • Latitude range: {coords_df['latitude'].min():.2f} to {coords_df['latitude'].max():.2f}"
    )
    print(
        f"   • Longitude range: {coords_df['longitude'].min():.2f} to {coords_df['longitude'].max():.2f}"
    )

print(f"\n🎯 FINAL SUMMARY:")
print(f"   • Total airports processed: {len(df_airports)}")
print(f"   • Coordinates found: {len(coords_df)}")
print(f"   • Success rate: {(len(coords_df)/len(df_airports)*100):.1f}%")
print(f"   • Data saved to: {output_file}")
print(f"   • DataFrame available as: df_airports")

💾 Saving airport coordinates to CSV...
✅ Saved to: ../data/airport_coordinates.csv

📋 COMPLETE AIRPORT COORDINATES TABLE:
iata_code  latitude   longitude    source
      1HR 60.318769   24.968125 nominatim
      1SN 60.318769   24.968125 nominatim
      3SK       NaN         NaN not_found
      AAL 57.086435    9.867982 nominatim
      AAR 56.304728   10.612299 nominatim
      ABZ 57.202227   -2.201736 nominatim
      ACE 28.945681  -13.607588 nominatim
      AES 62.560215    6.113434 nominatim
      AGP 36.678545   -4.499955 nominatim
      ALC 38.284418   -0.565753 nominatim
      ALF 69.977163   23.359519 nominatim
      AMS 52.326980    4.741505 nominatim
      ANX 69.291325   16.139681 nominatim
      AOK 35.421096   27.145333 nominatim
      ARN 59.646792   17.937044 nominatim
      AYT 36.899943   30.798191 nominatim
      BCN 41.296944    2.079047 nominatim
      BDU 69.054826   18.540665 nominatim
      BEG 44.818256   20.302360 nominatim
      BFS 34.131744 -101.423297 nomina

In [ ]:
# Quick preview of the final dataset
print("📊 FINAL DATASET PREVIEW:")
print("=" * 50)
print(df_airports.head(10))

print(f"\n🌍 COORDINATE VALIDATION:")
coords_df = df_airports[df_airports["latitude"].notna()]
print(
    f"   • Latitude range: {coords_df['latitude'].min():.2f}° to {coords_df['latitude'].max():.2f}°"
)
print(
    f"   • Longitude range: {coords_df['longitude'].min():.2f}° to {coords_df['longitude'].max():.2f}°"
)

print(f"\n✅ SUCCESS! Airport coordinates lookup complete:")
print(
    f"   📍 {len(coords_df)} out of {len(df_airports)} airports geocoded ({(len(coords_df)/len(df_airports)*100):.1f}% success rate)"
)
print(f"   💾 Data saved to: ../data/airport_coordinates.csv")
print(f"   📋 DataFrame available as: df_airports")

📊 FINAL DATASET PREVIEW:
  iata_code   latitude  longitude     source
0       1HR  60.318769  24.968125  nominatim
1       1SN  60.318769  24.968125  nominatim
2       3SK        NaN        NaN  not_found
3       AAL  57.086435   9.867982  nominatim
4       AAR  56.304728  10.612299  nominatim
5       ABZ  57.202227  -2.201736  nominatim
6       ACE  28.945681 -13.607588  nominatim
7       AES  62.560215   6.113434  nominatim
8       AGP  36.678545  -4.499955  nominatim
9       ALC  38.284418  -0.565753  nominatim

🌍 COORDINATE VALIDATION:
   • Latitude range: 14.29° to 71.03°
   • Longitude range: -118.42° to 116.59°

✅ SUCCESS! Airport coordinates lookup complete:
   📍 136 out of 140 airports geocoded (97.1% success rate)
   💾 Data saved to: ../data/airport_coordinates.csv
   📋 DataFrame available as: df_airports


## Using Premium Aviation API for Missing Airports
Using the aviation API key strategically to find coordinates for the 4 airports that couldn't be found with free APIs.

In [ ]:
# Use premium aviation API for the 4 missing airports only
AVIATION_API_KEY = "c2e2fa30ed2606064c984c208479d405"


def get_airport_coordinates_with_api_key(iata_code: str) -> Optional[Dict]:
    """
    Fetch airport coordinates using aviation-edge API with API key
    Returns dict with coordinates and source info or None if not found
    """
    try:
        url = f"https://aviation-edge.com/v2/public/airportDatabase"
        params = {"key": AVIATION_API_KEY, "iataCode": iata_code}

        print(f"🔑 Using premium API for {iata_code}...")
        response = requests.get(url, params=params, timeout=15)

        if response.status_code == 200:
            data = response.json()
            if data and len(data) > 0:
                airport_info = data[0]
                lat = airport_info.get("latitudeAirport")
                lon = airport_info.get("longitudeAirport")
                name = airport_info.get("nameAirport", "Unknown")

                if lat and lon:
                    lat = float(lat)
                    lon = float(lon)
                    if lat != 0 and lon != 0:
                        return {
                            "iata_code": iata_code,
                            "latitude": lat,
                            "longitude": lon,
                            "source": "aviation-edge-premium",
                            "airport_name": name,
                        }

        print(f"   ❌ No data found in premium API for {iata_code}")
        return None

    except Exception as e:
        print(f"   ⚠️ Premium API error for {iata_code}: {str(e)[:50]}")
        return None


# Get the missing airports (those without coordinates)
missing_airports = df_airports[df_airports["latitude"].isna()]["iata_code"].tolist()
print(
    f"🎯 Found {len(missing_airports)} airports without coordinates: {missing_airports}"
)
print(f"💰 Will use {len(missing_airports)} API requests out of 100 available")

if len(missing_airports) > 0:
    print(f"\n🔍 Looking up missing airports with premium API...")
    print("=" * 50)

    updated_airports = []
    api_requests_used = 0

    for airport_code in missing_airports:
        if api_requests_used >= 100:
            print(f"⚠️ Reached API request limit (100)")
            break

        premium_result = get_airport_coordinates_with_api_key(airport_code)
        api_requests_used += 1

        if premium_result:
            # Update the DataFrame with the new coordinates
            mask = df_airports["iata_code"] == airport_code
            df_airports.loc[mask, "latitude"] = premium_result["latitude"]
            df_airports.loc[mask, "longitude"] = premium_result["longitude"]
            df_airports.loc[mask, "source"] = premium_result["source"]

            updated_airports.append(airport_code)
            print(
                f"   ✅ {airport_code}: {premium_result['latitude']:.4f}, {premium_result['longitude']:.4f}"
            )
            print(f"      Airport: {premium_result['airport_name']}")

        time.sleep(1)  # Be respectful even with paid API

    print(f"\n📊 PREMIUM API RESULTS:")
    print(f"   • API requests used: {api_requests_used}")
    print(f"   • Airports found: {len(updated_airports)}")
    print(f"   • Still missing: {len(missing_airports) - len(updated_airports)}")

    if updated_airports:
        print(f"   • Updated airports: {', '.join(updated_airports)}")

else:
    print("✅ No missing airports - all coordinates already found!")

🎯 Found 4 airports without coordinates: ['3SK', 'SXF', 'VDB', 'VRY']
💰 Will use 4 API requests out of 100 available

🔍 Looking up missing airports with premium API...
🔑 Using premium API for 3SK...
   ⚠️ Premium API error for 3SK: 0
🔑 Using premium API for SXF...
   ⚠️ Premium API error for SXF: 0
🔑 Using premium API for VDB...
   ⚠️ Premium API error for VDB: 0
🔑 Using premium API for VRY...
   ⚠️ Premium API error for VRY: 0

📊 PREMIUM API RESULTS:
   • API requests used: 4
   • Airports found: 0
   • Still missing: 4


In [ ]:
# Save updated results with premium API data
print("💾 Saving updated airport coordinates...")

# Save to CSV file (overwrite previous version)
output_file = "../data/airport_coordinates.csv"
df_airports.to_csv(output_file, index=False)

# Display final statistics
coords_df = df_airports[df_airports["latitude"].notna()]
missing_df = df_airports[df_airports["latitude"].isna()]

print(f"\n🎯 FINAL RESULTS AFTER PREMIUM API:")
print("=" * 50)
print(f"   • Total airports: {len(df_airports)}")
print(f"   • With coordinates: {len(coords_df)}")
print(f"   • Still missing: {len(missing_df)}")
print(f"   • Success rate: {(len(coords_df)/len(df_airports)*100):.1f}%")

if len(missing_df) > 0:
    print(f"   • Remaining missing: {', '.join(missing_df['iata_code'].tolist())}")

# Show source breakdown
source_counts = df_airports["source"].value_counts()
print(f"\n🔍 DATA SOURCES USED:")
for source, count in source_counts.items():
    print(f"   • {source}: {count} airports")

print(f"\n✅ Updated data saved to: {output_file}")
print(f"📊 Ready for geospatial analysis and mapping!")

💾 Saving updated airport coordinates...

🎯 FINAL RESULTS AFTER PREMIUM API:
   • Total airports: 140
   • With coordinates: 136
   • Still missing: 4
   • Success rate: 97.1%
   • Remaining missing: 3SK, SXF, VDB, VRY

🔍 DATA SOURCES USED:
   • nominatim: 136 airports
   • not_found: 4 airports

✅ Updated data saved to: ../data/airport_coordinates.csv
📊 Ready for geospatial analysis and mapping!


In [ ]:
# Manual lookup for remaining 4 airports with expanded search terms
print("🔍 Manual lookup for remaining airports with expanded search terms...")

# Known airport names for better geocoding
airport_search_terms = {
    "3SK": ["3SK", "Scandinavian Airlines", "SAS"],
    "SXF": ["Berlin Schönefeld Airport", "Schönefeld", "SXF Airport Berlin"],
    "VDB": ["VDB Airport", "Västerås Airport", "Västerås"],
    "VRY": ["VRY Airport", "Värlby Airport", "Värlby"],
}


def enhanced_airport_search(iata_code: str, search_terms: list) -> Optional[Dict]:
    """Enhanced search using multiple search terms"""
    for term in search_terms:
        try:
            print(f"   🔍 Searching for: {term}")
            url = "https://nominatim.openstreetmap.org/search"
            params = {"q": term, "format": "json", "limit": 3, "addressdetails": 1}
            headers = {"User-Agent": "AirportAnalysis/1.0"}

            response = requests.get(url, params=params, headers=headers, timeout=10)

            if response.status_code == 200:
                data = response.json()
                for result in data:
                    # Look for airports specifically
                    display_name = result.get("display_name", "").lower()
                    if "airport" in display_name or "airfield" in display_name:
                        lat = float(result["lat"])
                        lon = float(result["lon"])
                        print(f"   ✅ Found airport match: {result['display_name']}")
                        return {
                            "iata_code": iata_code,
                            "latitude": lat,
                            "longitude": lon,
                            "source": "nominatim-enhanced",
                            "search_term": term,
                            "location_name": result["display_name"],
                        }

            time.sleep(1)  # Be respectful
        except Exception as e:
            print(f"   ⚠️ Error searching {term}: {str(e)[:30]}")

    return None


print("\nStarting enhanced search for remaining airports:")
print("=" * 50)

found_airports = []
for airport_code, search_terms in airport_search_terms.items():
    print(f"\n🔍 Enhanced search for {airport_code}:")

    result = enhanced_airport_search(airport_code, search_terms)
    if result:
        # Update the DataFrame
        mask = df_airports["iata_code"] == airport_code
        df_airports.loc[mask, "latitude"] = result["latitude"]
        df_airports.loc[mask, "longitude"] = result["longitude"]
        df_airports.loc[mask, "source"] = result["source"]

        found_airports.append(airport_code)
        print(
            f"   ✅ Updated {airport_code}: {result['latitude']:.4f}, {result['longitude']:.4f}"
        )
        print(f"   📍 Location: {result['location_name']}")
    else:
        print(f"   ❌ No airport found for {airport_code}")

print(f"\n📊 ENHANCED SEARCH RESULTS:")
print(f"   • Airports found: {len(found_airports)}")
if found_airports:
    print(f"   • Found: {', '.join(found_airports)}")

# Final statistics
coords_df = df_airports[df_airports["latitude"].notna()]
missing_df = df_airports[df_airports["latitude"].isna()]

print(f"\n🎯 OVERALL FINAL STATISTICS:")
print(f"   • Total airports: {len(df_airports)}")
print(f"   • With coordinates: {len(coords_df)}")
print(f"   • Success rate: {(len(coords_df)/len(df_airports)*100):.1f}%")

if len(missing_df) > 0:
    print(f"   • Still missing: {', '.join(missing_df['iata_code'].tolist())}")

🔍 Manual lookup for remaining airports with expanded search terms...

Starting enhanced search for remaining airports:

🔍 Enhanced search for 3SK:
   🔍 Searching for: 3SK
   🔍 Searching for: Scandinavian Airlines
   🔍 Searching for: SAS
   ❌ No airport found for 3SK

🔍 Enhanced search for SXF:
   🔍 Searching for: Berlin Schönefeld Airport
   ✅ Found airport match: Segro Airport Park Berlin, Bohnsdorf, Treptow-Köpenick, Berlin, Dahme-Spreewald, Brandenburg, 12526, Deutschland
   ✅ Updated SXF: 52.3944, 13.5384
   📍 Location: Segro Airport Park Berlin, Bohnsdorf, Treptow-Köpenick, Berlin, Dahme-Spreewald, Brandenburg, 12526, Deutschland

🔍 Enhanced search for VDB:
   🔍 Searching for: VDB Airport
   🔍 Searching for: Västerås Airport
   🔍 Searching for: Västerås
   ❌ No airport found for VDB

🔍 Enhanced search for VRY:
   🔍 Searching for: VRY Airport
   🔍 Searching for: Värlby Airport
   🔍 Searching for: Värlby
   ❌ No airport found for VRY

📊 ENHANCED SEARCH RESULTS:
   • Airports found: 

In [ ]:
# Save final results
print("\n💾 Saving final airport coordinates with all improvements...")

# Save to CSV file
output_file = "../data/airport_coordinates.csv"
df_airports.to_csv(output_file, index=False)

# Create a backup copy with timestamp for safety
backup_file = "../data/airport_coordinates_final.csv"
df_airports.to_csv(backup_file, index=False)

print(f"✅ Final data saved to: {output_file}")
print(f"✅ Backup saved to: {backup_file}")

# Final comprehensive summary
coords_df = df_airports[df_airports["latitude"].notna()]
missing_df = df_airports[df_airports["latitude"].isna()]

print(f"\n🎯 COMPREHENSIVE FINAL SUMMARY:")
print("=" * 60)
print(f"📊 STATISTICS:")
print(f"   • Total airports processed: {len(df_airports)}")
print(f"   • Successfully geocoded: {len(coords_df)}")
print(f"   • Could not find: {len(missing_df)}")
print(f"   • Overall success rate: {(len(coords_df)/len(df_airports)*100):.1f}%")

print(f"\n🔍 DATA SOURCES BREAKDOWN:")
source_counts = df_airports["source"].value_counts()
for source, count in source_counts.items():
    print(f"   • {source}: {count} airports")

print(f"\n🌍 COORDINATE COVERAGE:")
if len(coords_df) > 0:
    print(
        f"   • Latitude range: {coords_df['latitude'].min():.2f}° to {coords_df['latitude'].max():.2f}°"
    )
    print(
        f"   • Longitude range: {coords_df['longitude'].min():.2f}° to {coords_df['longitude'].max():.2f}°"
    )

if len(missing_df) > 0:
    print(f"\n❌ REMAINING UNRESOLVED AIRPORTS ({len(missing_df)}):")
    print(f"   • {', '.join(missing_df['iata_code'].tolist())}")
    print(f"   • These may be defunct airports, special codes, or very small airfields")

print(f"\n💰 API USAGE:")
print(f"   • Aviation API requests used: 4 out of 100 available")
print(f"   • Remaining requests: 96")

print(f"\n✅ SUCCESS! Airport coordinate analysis complete.")
print(f"📄 Data ready for:")
print(f"   • Flight route mapping")
print(f"   • Distance calculations")
print(f"   • Geospatial analysis")
print(f"   • Interactive visualizations")


💾 Saving final airport coordinates with all improvements...
✅ Final data saved to: ../data/airport_coordinates.csv
✅ Backup saved to: ../data/airport_coordinates_final.csv

🎯 COMPREHENSIVE FINAL SUMMARY:
📊 STATISTICS:
   • Total airports processed: 140
   • Successfully geocoded: 137
   • Could not find: 3
   • Overall success rate: 97.9%

🔍 DATA SOURCES BREAKDOWN:
   • nominatim: 136 airports
   • not_found: 3 airports
   • nominatim-enhanced: 1 airports

🌍 COORDINATE COVERAGE:
   • Latitude range: 14.29° to 71.03°
   • Longitude range: -118.42° to 116.59°

❌ REMAINING UNRESOLVED AIRPORTS (3):
   • 3SK, VDB, VRY
   • These may be defunct airports, special codes, or very small airfields

💰 API USAGE:
   • Aviation API requests used: 4 out of 100 available
   • Remaining requests: 96

✅ SUCCESS! Airport coordinate analysis complete.
📄 Data ready for:
   • Flight route mapping
   • Distance calculations
   • Geospatial analysis
   • Interactive visualizations
